In this example, we will see GEPA evolve the whole DSPy program (not just the instruction), including modifying the structure/dataflow of the program. We will use GEPA to tune a simple dspy.ChainOfThought module for MATH questions into a full DSPy program.

In [1]:
# Install python-dotenv and load .env.dev if present
try:
    from dotenv import load_dotenv
    import os
    if os.path.exists(".env.dev"):
        load_dotenv(".env.dev")
except ImportError:
    print("python-dotenv is not installed. Skipping .env.dev loading.")


In [2]:
import dspy

In [3]:
import random

from dspy.datasets import MATH

dataset = MATH(subset="algebra")

# Shuffle the train and dev sets
random.Random(0).shuffle(dataset.train)
random.Random(0).shuffle(dataset.dev)

print(len(dataset.train), len(dataset.dev), len(dataset.test))

/home/jacobjensen/Projects/prompt_evolution/gepa/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


350 350 487


Let's inspect an example from the training set.

In [4]:
example = dataset.train[0]
print("Question:", example.question)
print("Answer:", example.answer)

Question: If $f(x) = x^3 - 6x^2 + 3x - 4$, $g(x) = x^3 + 5x^2 + 9x - 2$, then find the constant term of $f(g(x))$.
Answer: -42


Let's define a simple DSPy program to solve this task.

Unlike dspy.GEPA that can take an instantiated DSPy module as input, here, we want to evolve the full DSPy program. Hence, a candidate here is the source code as string. The seed program does not need to be sophisticated, it just needs to demonstrate what the expected input/output interface is, and possibly the available tools. You can also include any additional information about the environment as a comment.

In [5]:
program_src = """import dspy
program = dspy.ChainOfThought("question -> answer")"""

GEPA interfaces with external frameworks through an adapter. In this case, we integrate GEPA with a DspyAdapter.

In [6]:
# 1) See which Python your kernel is using and the import path
import sys, platform
print("Python:", sys.version)
print("Executable:", sys.executable)
print("First 5 sys.path entries:")
print("\n".join(sys.path[:5]))

# 2) Check whether Python can even see the module chain you want
import importlib.util as ilu

def show_spec(name):
    spec = ilu.find_spec(name)
    print(f"{name:60} -> {'FOUND' if spec else 'NOT FOUND'}")
    if spec:
        print("  origin:", spec.origin)
        print("  submodule_search_locations:", spec.submodule_search_locations)

for m in [
    "gepa",
    "gepa.adapters",
    "gepa.adapters.dspy_full_program_adapter",
    "gepa.adapters.dspy_full_program_adapter.full_program_adapter",
    "dspy",
]:
    show_spec(m)

Python: 3.11.11 (main, Feb 12 2025, 14:51:05) [Clang 19.1.6 ]
Executable: /home/jacobjensen/Projects/prompt_evolution/gepa/.venv/bin/python
First 5 sys.path entries:
/home/jacobjensen/.local/share/uv/python/cpython-3.11.11-linux-x86_64-gnu/lib/python311.zip
/home/jacobjensen/.local/share/uv/python/cpython-3.11.11-linux-x86_64-gnu/lib/python3.11
/home/jacobjensen/.local/share/uv/python/cpython-3.11.11-linux-x86_64-gnu/lib/python3.11/lib-dynload

/home/jacobjensen/Projects/prompt_evolution/gepa/.venv/lib/python3.11/site-packages
gepa                                                         -> FOUND
  origin: /home/jacobjensen/Projects/prompt_evolution/gepa/src/gepa/__init__.py
  submodule_search_locations: ['/home/jacobjensen/Projects/prompt_evolution/gepa/src/gepa']
gepa.adapters                                                -> FOUND
  origin: /home/jacobjensen/Projects/prompt_evolution/gepa/src/gepa/adapters/__init__.py
  submodule_search_locations: ['/home/jacobjensen/Projects/prompt_

In [7]:


# 3) Enumerate what's inside gepa.adapters (using pkgutil)
import pkgutil, importlib, sys

gepa_adapters = importlib.import_module("gepa.adapters")
print("gepa.adapters.__path__ ->", list(gepa_adapters.__path__))

print("Subpackages/modules under gepa.adapters:")
for mod in pkgutil.iter_modules(gepa_adapters.__path__):
    print(" -", mod.name, "(package)" if mod.ispkg else "(module)")

gepa.adapters.__path__ -> ['/home/jacobjensen/Projects/prompt_evolution/gepa/src/gepa/adapters']
Subpackages/modules under gepa.adapters:
 - anymaths_adapter (package)
 - default_adapter (package)
 - dspy_adapter (package)
 - dspy_full_program_adapter (package)
 - ifbench_adapter (package)
 - terminal_bench_adapter (package)


In [8]:
from gepa.adapters.dspy_full_program_adapter.full_program_adapter import DspyAdapter

In [9]:
def metric_fn(example, pred, trace=None):
    score = dataset.metric(example, pred)
    if score:
        feedback_text = f"The provided answer '{pred.answer}' is correct."
    else:
        feedback_text = f"The provided answer '{pred.answer}' is incorrect. The correct answer is '{example.answer}'. Here's the step by step solution:\n{example.reasoning}"
    return dspy.Prediction(score=score, feedback=feedback_text)

In [10]:
reflection_lm = dspy.LM(model="openai/gpt-4.1", max_tokens=32000)  # temperature=1
adapter = DspyAdapter(
    task_lm=dspy.LM(model="openai/gpt-4.1-nano", max_tokens=32000),
    metric_fn=metric_fn,
    num_threads=80,
    reflection_lm=lambda x: reflection_lm(x)[0],
)

Let's evaluate the base program

In [11]:
o = adapter.evaluate(dataset.test, {"program": program_src})

2025/09/13 16:34:54 INFO dspy.evaluate.evaluate: Average Metric: 328.0 / 487 (67.4%)


The base program obtains a score of 67.1%

Let's launch the GEPA optimization.

In [12]:
from gepa import optimize

o = optimize(
    seed_candidate={"program": program_src},
    trainset=dataset.train,
    valset=dataset.dev[:200],
    adapter=adapter,
    reflection_lm=lambda x: reflection_lm(x)[0],
    max_metric_calls=2000,
    display_progress_bar=True,
)

GEPA Optimization:  10%|█         | 200/2000 [00:21<03:13,  9.30rollouts/s]

Iteration 0: Base program full valset score: 0.66
Iteration 1: Selected program 0 score: 0.66
Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:04<00:00,  1.61s/it]

2025/09/13 16:35:20 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)



Iteration 1: Proposed new text for program: import dspy
from typing import Literal

class MathQA(dspy.Signature):
    """
    Solve the given math word problem step by step, showing all necessary reasoning and calculations.
    Then, extract and return ONLY the final numeric answer as a string, with no units or extra text.
    
    Instructions:
    - Show clear, step-by-step reasoning in the 'reasoning' field.
    - In the 'answer' field, provide ONLY the final numeric answer (e.g., '42', '6', '32'), not a sentence or explanation.
    - Do not include units, words, or extra formatting in the answer.
    - If there are multiple valid numeric answers, sum them and return the sum as a string.
    - Common pitfalls: Do not restate the question or include sentences in the answer field. Do not include units or box the answer.
    - Example:
        Question: "If x + 2 = 5, what is x?"
        Reasoning: "x + 2 = 5 => x = 5 - 2 = 3"
        Answer: "3"
    - Example:
        Question: "Let 

2025/09/13 16:35:35 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/13 16:37:58 INFO dspy.evaluate.evaluate: Average Metric: 155.0 / 200 (77.5%)
GEPA Optimization:  20%|██        | 406/2000 [03:04<13:39,  1.94rollouts/s]

...


Iteration 41: All subsample scores perfect. Skipping.
Iteration 41: Reflective mutation did not propose a new candidate
Iteration 42: Selected program 4 score: 0.825
Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:08<00:00,  2.83s/it]

2025/09/13 17:03:04 INFO dspy.evaluate.evaluate: Average Metric: 1.0 / 3 (33.3%)



Iteration 42: Proposed new text for program: import dspy
from typing import Optional

class MathQAReasoning(dspy.Signature):
    """
    Solve the given math word problem step by step, showing all necessary reasoning and calculations.
    Do NOT provide the final answer yet; focus only on the step-by-step solution and intermediate results.

    Instructions:
    - Show clear, step-by-step reasoning in the 'reasoning' field.
    - Do NOT include the final answer or summary sentence.
    - Use mathematical notation (e.g., fractions, radicals, $\pi$, $\LaTeX$) as appropriate.
    - Avoid restating the question.
    - Example:
        Question: "What is the value of $\\frac{1}{2} + \\frac{1}{3}$? Express your answer as a common fraction."
        Reasoning: "Find common denominator: 1/2 + 1/3 = 3/6 + 2/6 = 5/6."
    """
    question: str = dspy.InputField(desc="The math word problem to solve")
    reasoning: str = dspy.OutputField(desc="Step-by-step solution and calculations, no final ans

2025/09/13 17:03:30 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2025/09/13 17:04:13 INFO dspy.evaluate.evaluate: Average Metric: 176.0 / 200 (88.0%)
GEPA Optimization: 100%|█████████▉| 1998/2000 [29:18<00:01,  1.14rollouts/s]

Iteration 42: Full valset score for new program: 0.88
Iteration 42: Full train_val score for new program: 0.88
Iteration 42: Individual valset scores for new program: [True, False, True, True, False, True, True, True, False, True, True, True, True, True, True, False, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, False, True, True, False, True, False, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, False, False, True, True, True, True, False, True, False, True, True, True, True, True, True, True, True, True, False, True, True, True, True, True, True, True, Tru

Let's see the DSPy program found by GEPA

In [13]:
print(o.best_candidate["program"])

import dspy
import re
from typing import Optional

class MathQA(dspy.Signature):
    """
    Solve the given math word problem step by step, showing all necessary reasoning and calculations.
    Then, extract and return ONLY the final answer in the required format (number, variable, coordinate, interval, or list), with no units or extra text.

    Instructions:
    - Show clear, step-by-step reasoning in the 'reasoning' field.
    - In the 'answer' field, provide ONLY the final answer in the format required by the question:
        - If the answer is a number, return just the number (e.g., '42', '6', '0').
        - If the answer is a coordinate, return it in the form '(x,y)' (e.g., '(9,11)').
        - If the answer is a variable or object, return just its name (e.g., 'x', 'A').
        - If the answer is a list or set, return the elements separated by spaces (e.g., '2 3 5'), unless the question specifies a sum.
        - If the answer is an interval or union of intervals, use standar

Evaluating the optimized program

In [14]:
_ = adapter.evaluate(dataset.test, o.best_candidate)

2025/09/13 17:05:17 INFO dspy.evaluate.evaluate: Average Metric: 414.0 / 487 (85.0%)


We see it going from **67% to 93%** in just a few rounds of optimization!